# Technical Analysis Ingestion Test

This notebook tests and validates the technical analysis calculation pipeline:
- Checks current analysis coverage vs market_prices
- Identifies missing analysis records
- Verifies indicator calculations (EMA, RSI, MACD, ATR)
- Generates commands to fill missing analysis
- Validates data integrity

**Date**: 2026-08-26  
**Database**: fin-market-db (Azure SQL)  
**Schema**: crypto.technical_analysis

## 1. Setup and Imports

In [1]:
import sys
from pathlib import Path
from datetime import datetime, timedelta, timezone
import pandas as pd
from sqlalchemy import create_engine, text, func
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

from cryptoquant.database.session import get_session
from cryptoquant.database.models import MarketPrice, TechnicalAnalysis, TradingPair
from cryptoquant.config.settings import get_settings

print("✓ Imports successful")
print(f"Current time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")

✓ Imports successful
Current time: 2026-08-27 01:47:18 UTC


## 2. Database Connection

In [ ]:
session = get_session()
settings = get_settings()

print(f"✓ Connected to: {settings.database_url}")

print(f"✓ Schema: crypto")

AttributeError: 'Settings' object has no attribute 'db_server'

## 3. Current Analysis Overview

In [ ]:
query = text("""
SELECT 
    COUNT(*) AS total_analysis_records,
    MIN(timestamp) AS first_analysis,
    MAX(timestamp) AS latest_analysis,
    DATEDIFF(DAY, MIN(timestamp), MAX(timestamp)) AS days_span,
    COUNT(DISTINCT trading_pair_id) AS distinct_pairs,
    COUNT(CASE WHEN ema_200 IS NOT NULL THEN 1 END) AS records_with_ema,
    COUNT(CASE WHEN rsi_14 IS NOT NULL THEN 1 END) AS records_with_rsi,
    COUNT(CASE WHEN macd IS NOT NULL THEN 1 END) AS records_with_macd,
    COUNT(CASE WHEN atr_14 IS NOT NULL THEN 1 END) AS records_with_atr
FROM crypto.technical_analysis
""")

result = session.execute(query).fetchone()
df_overview = pd.DataFrame([result._mapping], columns=result._mapping.keys())

print("=== Technical Analysis Overview ===")
display(df_overview)

total = result.total_analysis_records
print(f"\n📊 Total analysis records: {total:,}")

## 4. Analysis Coverage by Trading Pair

Compare analysis records to market_prices (after 200h warm-up for EMA 200):

In [ ]:
query = text("""
WITH warmup_threshold AS (
    SELECT
        tp.id AS trading_pair_id,
        tp.symbol,
        MIN(mp.timestamp) AS first_candle,
        DATEADD(HOUR, 200, MIN(mp.timestamp)) AS warmup_complete
    FROM crypto.market_prices AS mp
    INNER JOIN crypto.trading_pairs AS tp ON tp.id = mp.trading_pair_id
    GROUP BY tp.id, tp.symbol
)
SELECT
    tp.symbol AS currency_pair,
    COUNT(mp.id) AS market_prices_after_warmup,
    COUNT(ta.id) AS analysis_records,
    COUNT(mp.id) - COUNT(ta.id) AS missing_analysis,
    CAST(CASE
        WHEN COUNT(mp.id) = 0 THEN 0
        ELSE COUNT(ta.id) * 100.0 / COUNT(mp.id)
    END AS DECIMAL(5,2)) AS coverage_percent,
    w.warmup_complete,
    MAX(ta.timestamp) AS latest_analysis,
    DATEDIFF(HOUR, MAX(ta.timestamp), SYSUTCDATETIME()) AS hours_since_latest
FROM crypto.trading_pairs AS tp
INNER JOIN warmup_threshold AS w ON w.trading_pair_id = tp.id
LEFT JOIN crypto.market_prices AS mp 
    ON mp.trading_pair_id = tp.id 
    AND mp.timestamp >= w.warmup_complete
LEFT JOIN crypto.technical_analysis AS ta ON ta.market_price_id = mp.id
GROUP BY tp.symbol, w.warmup_complete
ORDER BY coverage_percent ASC
""")

df_coverage = pd.DataFrame(session.execute(query).fetchall(),
                           columns=['currency_pair', 'market_prices_after_warmup', 'analysis_records',
                                   'missing_analysis', 'coverage_percent', 'warmup_complete',
                                   'latest_analysis', 'hours_since_latest'])

print("=== Analysis Coverage by Trading Pair ===")
display(df_coverage)

# Highlight issues
issues = df_coverage[df_coverage['coverage_percent'] < 98.0]
if len(issues) > 0:
    print(f"\n⚠️  {len(issues)} pair(s) with < 98% coverage - needs attention")
else:
    print("\n✓ All pairs have >= 98% analysis coverage")

## 5. Identify Analysis Gaps (> 1 hour)

In [ ]:
query = text("""
WITH ordered_analysis AS (
    SELECT
        ta.trading_pair_id,
        tp.symbol,
        ta.timestamp,
        LEAD(ta.timestamp) OVER (PARTITION BY ta.trading_pair_id ORDER BY ta.timestamp) AS next_timestamp
    FROM crypto.technical_analysis AS ta
    INNER JOIN crypto.trading_pairs AS tp ON tp.id = ta.trading_pair_id
),
gaps AS (
    SELECT
        symbol,
        DATEDIFF(HOUR, timestamp, next_timestamp) AS hours_gap
    FROM ordered_analysis
    WHERE next_timestamp IS NOT NULL
      AND DATEDIFF(HOUR, timestamp, next_timestamp) > 1
)
SELECT
    symbol AS currency_pair,
    COUNT(*) AS gap_count,
    SUM(hours_gap) AS total_hours_missing,
    MIN(hours_gap) AS min_gap_hours,
    MAX(hours_gap) AS max_gap_hours,
    AVG(hours_gap) AS avg_gap_hours
FROM gaps
GROUP BY symbol
ORDER BY total_hours_missing DESC
""")

df_gaps = pd.DataFrame(session.execute(query).fetchall(),
                       columns=['currency_pair', 'gap_count', 'total_hours_missing',
                               'min_gap_hours', 'max_gap_hours', 'avg_gap_hours'])

if len(df_gaps) > 0:
    print("=== Analysis Gaps Summary ===")
    display(df_gaps)
    total_missing = df_gaps['total_hours_missing'].sum()
    print(f"\n⚠️  Total missing analysis hours across all pairs: {total_missing:,.0f}")
else:
    print("✓ No gaps detected - all hourly analysis present!")

## 6. Sample Recent Analysis Data

Verify indicator calculations are present:

In [ ]:
query = text("""
SELECT TOP 10
    tp.symbol,
    ta.timestamp,
    CAST(ta.ema_200 AS DECIMAL(18,2)) AS ema_200,
    CAST(ta.rsi_14 AS DECIMAL(5,2)) AS rsi_14,
    CAST(ta.macd AS DECIMAL(18,8)) AS macd,
    CAST(ta.macd_signal AS DECIMAL(18,8)) AS macd_signal,
    CAST(ta.atr_14 AS DECIMAL(18,8)) AS atr_14,
    ta.signal
FROM crypto.technical_analysis AS ta
INNER JOIN crypto.trading_pairs AS tp ON tp.id = ta.trading_pair_id
WHERE tp.symbol = 'BTC-USD'
ORDER BY ta.timestamp DESC
""")

df_sample = pd.DataFrame(session.execute(query).fetchall(),
                        columns=['symbol', 'timestamp', 'ema_200', 'rsi_14', 'macd',
                                'macd_signal', 'atr_14', 'signal'])

print("=== Recent BTC-USD Analysis (Last 10) ===")
display(df_sample)

# Validation checks
if len(df_sample) > 0:
    print("\n✓ Indicator Validation:")
    print(f"  EMA 200 range: {df_sample['ema_200'].min():.2f} - {df_sample['ema_200'].max():.2f}")
    print(f"  RSI 14 range: {df_sample['rsi_14'].min():.2f} - {df_sample['rsi_14'].max():.2f} (should be 0-100)")
    print(f"  MACD range: {df_sample['macd'].min():.8f} - {df_sample['macd'].max():.8f}")
    print(f"  ATR 14 range: {df_sample['atr_14'].min():.8f} - {df_sample['atr_14'].max():.8f}")
else:
    print("⚠️  No analysis data found for BTC-USD")

## 7. Generate Re-Calculation Commands

Based on coverage analysis, generate PowerShell commands to fill missing analysis:

In [ ]:
print("=== Re-Calculation Commands ===\n")
print("Copy and paste these commands in PowerShell:\n")
print("cd D:\\data\\development\\crypto")
print(".\.venv\Scripts\Activate.ps1\n")

for idx, row in df_coverage.iterrows():
    pair = row['currency_pair']
    coverage = row['coverage_percent']
    missing = row['missing_analysis']
    
    if coverage < 50:
        # Critical - full historical calculation
        priority = "HIGH PRIORITY"
        mode = "historical"
        days = 730
    elif coverage < 90:
        # Medium - historical mode
        priority = "MEDIUM"
        mode = "historical"
        days = 365
    elif coverage < 98:
        # Low - incremental should catch up
        priority = "LOW"
        mode = "incremental"
        days = None
    else:
        continue  # Good coverage, skip
    
    print(f"# {priority}: {pair} ({coverage}% coverage, {missing} missing)")
    if mode == "incremental":
        print(f"python scripts/calculate_technical_analysis.py --mode incremental --pair {pair}")
    else:
        print(f"python scripts/calculate_technical_analysis.py --mode historical --days {days} --pair {pair}")
    print()

if df_coverage['coverage_percent'].min() >= 98:
    print("✓ All pairs have >= 98% coverage - No re-calculation needed!")

## 8. Historical Analysis Commands (All Pairs)

In [ ]:
print("=== Full Historical Analysis Commands ===\n")
print("# 30 days (all pairs)")
print("python scripts/calculate_technical_analysis.py --mode historical --days 30\n")

print("# 90 days (all pairs)")
print("python scripts/calculate_technical_analysis.py --mode historical --days 90\n")

print("# 1 year (all pairs)")
print("python scripts/calculate_technical_analysis.py --mode historical --days 365\n")

print("# 2 years (all pairs)")
print("python scripts/calculate_technical_analysis.py --mode historical --days 730\n")

print("# Incremental (all pairs)")
print("python scripts/calculate_technical_analysis.py --mode incremental\n")

## 9. Complete Pipeline Commands

Run candles + analysis together:

In [ ]:
print("=== Complete Ingestion + Analysis Pipeline ===\n")
print("PowerShell script to run both candles and analysis:")
print("-" * 60)
print("""# Activate environment
cd D:\\data\\development\\crypto
.\\.venv\\Scripts\\Activate.ps1

# Step 1: Ingest candles (2 years)
Write-Host "`n=== Step 1: Ingesting Candles ===" -ForegroundColor Cyan
python scripts/collect_historic_data.py --granularity hourly --days 730

# Step 2: Calculate technical analysis
Write-Host "`n=== Step 2: Calculating Technical Analysis ===" -ForegroundColor Cyan
python scripts/calculate_technical_analysis.py --mode historical --days 730

# Step 3: Verify results
Write-Host "`n=== Step 3: Verification ===" -ForegroundColor Cyan
python scripts/verify_technical_analysis.py --pair BTC-USD

Write-Host "`n✓ Pipeline complete!" -ForegroundColor Green
""")

## 10. Validation Summary

In [ ]:
print("=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

total_analysis = df_overview['total_analysis_records'][0]
min_coverage = df_coverage['coverage_percent'].min()
avg_coverage = df_coverage['coverage_percent'].mean()
pairs_with_issues = len(df_coverage[df_coverage['coverage_percent'] < 98.0])

print(f"\n📊 Total Analysis Records: {total_analysis:,}")
print(f"📈 Coverage Range: {min_coverage:.2f}% - 100%")
print(f"📈 Average Coverage: {avg_coverage:.2f}%")
print(f"⚠️  Pairs Needing Attention: {pairs_with_issues}")

if len(df_gaps) > 0:
    print(f"⚠️  Total Gaps: {len(df_gaps)} pair(s) have gaps > 1 hour")
else:
    print("✓ No gaps detected")

# Indicator validation
ema_coverage = (df_overview['records_with_ema'][0] / total_analysis * 100) if total_analysis > 0 else 0
rsi_coverage = (df_overview['records_with_rsi'][0] / total_analysis * 100) if total_analysis > 0 else 0
macd_coverage = (df_overview['records_with_macd'][0] / total_analysis * 100) if total_analysis > 0 else 0
atr_coverage = (df_overview['records_with_atr'][0] / total_analysis * 100) if total_analysis > 0 else 0

print(f"\n📊 Indicator Coverage:")
print(f"  EMA 200: {ema_coverage:.1f}%")
print(f"  RSI 14: {rsi_coverage:.1f}%")
print(f"  MACD: {macd_coverage:.1f}%")
print(f"  ATR 14: {atr_coverage:.1f}%")

print("\n" + "=" * 70)

if min_coverage >= 98.0 and len(df_gaps) == 0 and ema_coverage >= 99:
    print("✅ STATUS: EXCELLENT - Analysis is complete and accurate")
elif min_coverage >= 95.0:
    print("⚠️  STATUS: GOOD - Minor gaps, low priority fixes needed")
elif min_coverage >= 90.0:
    print("⚠️  STATUS: NEEDS ATTENTION - Medium priority re-calculation needed")
else:
    print("❌ STATUS: CRITICAL - Major gaps, high priority re-calculation needed")

print("=" * 70)

## 11. Close Connection

In [ ]:
session.close()
print("✓ Database connection closed")